In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re

import sys
sys.path.append('../../..')
# from mount_drive import mount_s_drive

In [2]:
# mount_s_drive(subfolder='LCICM/Databases/eICU')

In [3]:
myHours = 60 * 6

In [4]:
database_folder = '/projects/LCICM/eICU/'

### Patients 

In [5]:
patients_df = pd.read_csv(database_folder+'patient.csv')

In [6]:
patients_df.columns

Index(['patientunitstayid', 'patienthealthsystemstayid', 'gender', 'age',
       'ethnicity', 'hospitalid', 'wardid', 'apacheadmissiondx',
       'admissionheight', 'hospitaladmittime24', 'hospitaladmitoffset',
       'hospitaladmitsource', 'hospitaldischargeyear',
       'hospitaldischargetime24', 'hospitaldischargeoffset',
       'hospitaldischargelocation', 'hospitaldischargestatus', 'unittype',
       'unitadmittime24', 'unitadmitsource', 'unitvisitnumber', 'unitstaytype',
       'admissionweight', 'dischargeweight', 'unitdischargetime24',
       'unitdischargeoffset', 'unitdischargelocation', 'unitdischargestatus',
       'uniquepid'],
      dtype='object')

In [7]:
myIds = pd.read_csv('patient_ids.csv')
myIds

,Unnamed: 0,patientunitstayid
0,93,141816
1,220,142723
2,255,143056
3,601,145204
4,602,145205
...,...,...
4269,200720,3352475
4270,200731,3352512
4271,200778,3352801
4272,200844,3353194


In [8]:
patients_df = patients_df[patients_df['patientunitstayid'].isin(myIds.patientunitstayid)]

In [9]:
# patients_df[patients_df['apacheadmissiondx'].astype(str).str.contains('Cardiac arrest')].patientunitstayid.nunique()

In [10]:
# myIds = patients_df[(patients_df['age'].replace({'> 89': 89}).fillna(0).astype(int) >= 18) & 
#             (patients_df['apacheadmissiondx'].astype(str).str.contains('Cardiac arrest '))].patientunitstayid

In [11]:
myPredictorsDf = patients_df[['patientunitstayid', 'gender', 'age', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitsource', 'admissionweight']]

### Treatments

In [12]:
def getOneHotConditions(aDf, aColumn, aPrefix):
    aDf['conditions'] = aDf[aColumn].str.split('|')
    one_hot = aDf['conditions'].explode().str.get_dummies().groupby(level=0).sum()
    one_hot = one_hot.add_prefix(aPrefix)
    aDf = aDf.drop(columns=['conditions']).join(one_hot)
    return aDf, one_hot

In [13]:
treatment_df = pd.read_csv(database_folder+'treatment.csv')
treatment_df = treatment_df[treatment_df['patientunitstayid'].isin(myIds.patientunitstayid)]

In [14]:
treatment_df_hyp_time = (
    treatment_df[treatment_df['treatmentstring'].astype(str).str.contains('hypothermia')] \
    .groupby('patientunitstayid').agg('first')
)

In [15]:
treatment_df_hyp_time['treatmentoffset']

patientunitstayid
250724      1256
251469       655
252908       149
252954      2531
256864       975
           ...  
3352203     1970
3352445      451
3352512     5600
3353194       52
3353251    11304
Name: treatmentoffset, Length: 793, dtype: int64

In [16]:
treatment_df, one_hot = getOneHotConditions(treatment_df, 'treatmentstring', 'treatment_')

In [17]:
filtered_df = treatment_df[(treatment_df.treatmentoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
filtered_df_hyp = treatment_df.groupby('patientunitstayid').agg('sum').reset_index()

In [18]:
filtered_df[one_hot.columns] = (filtered_df[one_hot.columns] != 0).astype(int)
filtered_df_hyp['hyp'] = (filtered_df_hyp.treatment_hypothermia != 0).astype(int)
filtered_df_hyp.hyp.sum()

793

In [19]:
myPredictorsDf1 = myPredictorsDf.merge(filtered_df[['patientunitstayid'] + list(one_hot.columns)], on=['patientunitstayid'], how='left')
myPredictorsDf1 = myPredictorsDf1.merge(filtered_df_hyp[['patientunitstayid', 'hyp']], on=['patientunitstayid'], how='left')
myPredictorsDf1['treatment_hypothermia'] = myPredictorsDf1.hyp
myPredictorsDf1.drop(columns=['hyp'], inplace = True)

In [20]:
myPredictorsDf1 = myPredictorsDf.merge(treatment_df_hyp_time, on=['patientunitstayid'], how='left')

In [21]:
myPredictorsDf1['hypothermia_time'] = myPredictorsDf1['treatmentoffset']

In [22]:
myPredictorsDf = myPredictorsDf1

### Diagnosis

In [23]:
diagnosis_df = pd.read_csv(database_folder+'diagnosis.csv')
diagnosis_df = diagnosis_df[diagnosis_df['patientunitstayid'].isin(myIds.patientunitstayid)]

In [24]:
# myIds = diagnosis_df[diagnosis_df.icd9code.astype(str).str.contains('427.5')].patientunitstayid.unique()

In [25]:
diagnosis_df[diagnosis_df.icd9code.astype(str).str.contains('427.5')]

,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
350,4249756,141816,True,35,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
895,3872672,142723,True,143,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
1033,4171191,143056,False,7,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
1040,3838149,143056,True,900,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
2309,3975271,145204,False,3,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Other
...,...,...,...,...,...,...,...
2710633,46086056,3353251,False,4080,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
2710643,46087802,3353251,False,2653,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Other
2710645,46085958,3353251,False,5521,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
2710646,46086896,3353251,False,9169,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary


In [26]:
diagnosis_df, one_hot = getOneHotConditions(diagnosis_df, 'diagnosisstring', 'diagnosis_')

In [27]:
filtered_df = diagnosis_df[(diagnosis_df.diagnosisoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
filtered_df[one_hot.columns] = (filtered_df[one_hot.columns] != 0).astype(int)

In [28]:
myPredictorsDf1 = myPredictorsDf.merge(filtered_df[['patientunitstayid'] + list(one_hot.columns)], on=['patientunitstayid'], how='left')

In [29]:
# diagnosis_df['diagnosisstring'].contains('hypothermia'

In [30]:
myPredictorsDf = myPredictorsDf1

### Nurse Charting

In [ ]:
file_path = database_folder+'nurseCharting.csv'
chunk_size = 1e6

df_chunks = []

num_chunks = 0
for chunk in pd.read_csv(file_path,chunksize=chunk_size):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(myIds.patientunitstayid)]
    df_chunks.append(filtered_chunk)
    if num_chunks % 10 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk

nurse_charting_df = pd.concat(df_chunks, ignore_index=True)
print('Processing finished.')

Chunk 0 Processed.


### Hypothermmia Analysis

In [ ]:
myDfTempC = nurse_charting_df[(nurse_charting_df.nursingchartcelltypevalname == 'Temperature (C)')]
myDfTempC.nursingchartvalue = myDfTempC.nursingchartvalue.astype(float)
myDfTempC.sort_values(by=['patientunitstayid', 'nursingchartentryoffset'], inplace=True)
# myDfTempC['TimeDiff'] = myDfTempC['nursingchartoffset'].diff()
# myDfTempC['TempTimesTime'] = myDfTempC['TimeDiff'] * myDfTempC['nursingchartvalue'].shift(1)
# myDfTempC['RollingTemp'] = myDfTempC['TempTimesTime'].rolling(window=2).sum()
# myDfTempC['RollingTime'] = myDfTempC['TimeDiff'].rolling(window=2).sum()
# myDfTempC['AvgTemp'] = myDfTempC['RollingTemp'] / myDfTempC['RollingTime']
# myDfTempC[['TempTimesTime', 'TimeDiff', 'nursingchartvalue', 'RollingTemp', 'RollingTime', 'AvgTemp']]
myDfTempC2 = myDfTempC[(myDfTempC.nursingchartentryoffset < 2880)  \
        & (myDfTempC.nursingchartentryoffset > 0)
        & (myDfTempC.nursingchartvalue < 36)]
myDfTempC2['nursingchartentryoffset2'] = myDfTempC2['nursingchartentryoffset']
myHyp = myDfTempC2.groupby('patientunitstayid').agg({'patientunitstayid':'count', 'nursingchartvalue': 'min', 'nursingchartentryoffset':'min', 'nursingchartentryoffset2':'max'})
myHyp['Hyp'] = (myHyp['patientunitstayid'] > 12).astype(int) \
                & (myHyp['nursingchartvalue'] > 25).astype(int) \
                & (myHyp['nursingchartentryoffset2'] - myHyp['nursingchartentryoffset'] > 720).astype(int)
myHyp[(myHyp['Hyp'] == 1)] 

In [ ]:
myPredictorsDf['Hypothermia'] = 0
myPredictorsDf.loc[myPredictorsDf.patientunitstayid.isin(myHyp[myHyp['Hyp'] == 1].index), 'Hypothermia'] = 1
myPredictorsDf[['Hypothermia', 'patientunitstayid']]
myPredictorsDf.Hypothermia.sum()
# myPredictorsDf.drop(columns='Hypothermia', inplace=True)

In [ ]:
myPredictorsDf['both_hypothermia'] = ((myPredictorsDf.treatment_hypothermia == 1) | (myPredictorsDf.Hypothermia == 1)).astype(int)

In [ ]:
(myDfTempC[(myDfTempC.patientunitstayid ==3243869) & (myDfTempC.nursingchartentryoffset < 500)])

### Getting Features 

In [ ]:
def getFeaturesFromDf(aDf, aTimeColumn, aTypeColumn, aValueColumn):
    aDf['num_values'] = pd.to_numeric(aDf[aValueColumn], errors='coerce')
    aDf = aDf[['patientunitstayid', aTypeColumn, 'num_values', aTimeColumn]].copy()
    myMasterGroupDf = aDf[(aDf[aTimeColumn] < myHours)].groupby(['patientunitstayid', aTypeColumn])
    myMasterGroupBegin = aDf.loc[myMasterGroupDf[aTimeColumn].idxmin()]
    myMasterGroupEnd = aDf.loc[myMasterGroupDf[aTimeColumn].idxmax()]
    myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})
    myMasterGroupAgg = myMasterGroupAgg.num_values.reset_index()
    return myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg

In [ ]:
def mergeFeaturesInDf(aDfToMerge, aBegin, aEnd, aAgg, aPrefix, aTypeColumn, aValueColumn):
    print('Running Begin Columns')
    total = int(aBegin[aTypeColumn].nunique() / 10)
    myPredictorsDf1 = aDfToMerge
    i = 0
    print('progress: ', end="")
    for value in aBegin[aTypeColumn].unique():
        myDf = aBegin[aBegin[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', aValueColumn]], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_first_' + value] = myPredictorsDf1[aValueColumn]
        myPredictorsDf1.drop(columns=[aValueColumn], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    print()
    print('Running End Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    i = 0
    print('progress: ', end="")
    for value in aEnd[aTypeColumn].unique():
        myDf = aEnd[aEnd[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', aValueColumn]], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_last_' + value] = myPredictorsDf1[aValueColumn]
        myPredictorsDf1.drop(columns=[aValueColumn], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    print()
    print('Running Agg Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    i = 0
    print('progress: ', end="")
    for value in aAgg[aTypeColumn].unique():
        myDf = aAgg[aAgg[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', 'max', 'min', 'mean']], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
        myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
        myPredictorsDf1[aPrefix + '_mean_' + value] = myPredictorsDf1['mean']
        myPredictorsDf1.drop(columns=['max', 'min', 'mean'], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    return myPredictorsDf1

In [ ]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(nurse_charting_df, 'nursingchartoffset', 'nursingchartcelltypevalname', 'num_values')

In [ ]:
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'nurse', 'nursingchartcelltypevalname', 'num_values')

In [ ]:
print(list(myPredictorsDf.columns))

In [ ]:
myPredictorsDf = myPredictorsDf1

In [ ]:
gcs_df = nurse_charting_df[nurse_charting_df['nursingchartcelltypevalname']=='GCS Total']

In [ ]:
motor_gcs_df = nurse_charting_df[nurse_charting_df.nursingchartcelltypevalname == 'Motor']

In [ ]:
myGcsDf = gcs_df.sort_values(by=['patientunitstayid', "nursingchartoffset"])
myGcsDf['nursingchartoffset2'] = myGcsDf.nursingchartoffset
myGcsDf2 = myGcsDf.groupby('patientunitstayid').agg({'nursingchartoffset':'min', 'nursingchartoffset2': 'max'})
myGcsDfMin = myGcsDf.merge(myGcsDf2, on=['patientunitstayid', 'nursingchartoffset'], how='inner')
myGcsDfMax = myGcsDf.merge(myGcsDf2, on=['patientunitstayid', 'nursingchartoffset2'], how='inner')

In [ ]:
myMGcsDf = motor_gcs_df.sort_values(by=['patientunitstayid', 'nursingchartoffset'])
myMGcsDf['nursingchartoffset2'] = myMGcsDf.nursingchartoffset
myMGcsDf2 = myMGcsDf.groupby('patientunitstayid').agg({'nursingchartoffset':'min', 'nursingchartoffset2': 'max'})
myMGcsDfMin = myMGcsDf.merge(myMGcsDf2, on=['patientunitstayid', 'nursingchartoffset'], how='inner')
myMGcsDfMax = myMGcsDf.merge(myMGcsDf2, on=['patientunitstayid', 'nursingchartoffset2'], how='inner')

In [ ]:
myMGcsDfMax

In [ ]:
myPredictorsDf = myPredictorsDf.merge(myGcsDfMin[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'FirstGCSTime', 'nursingchartvalue': 'FirstGCS'}, inplace=True)
myPredictorsDf =myPredictorsDf.merge(myGcsDfMax[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'LastGCSTime', 'nursingchartvalue': 'LastGCS'}, inplace=True)

In [ ]:
myPredictorsDf = myPredictorsDf.merge(myMGcsDfMin[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid', how='outer')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'FirstMGCSTime', 'nursingchartvalue': 'FirstMGCS'}, inplace=True)
myPredictorsDf =myPredictorsDf.merge(myMGcsDfMax[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid', how='outer')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'LastMGCSTime', 'nursingchartvalue': 'LastMGCS'}, inplace=True)

### Labs

In [ ]:
file_path = database_folder+'lab.csv'
chunk_size = 1e6

# patient_unit_stay_ids = ca_patients_df['patientunitstayid']
df_chunks = []

num_chunks = 0
for chunk in pd.read_csv(file_path,chunksize=chunk_size):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(myIds.patientunitstayid)]
    df_chunks.append(filtered_chunk)
    if num_chunks % 10 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk

myLabs = pd.concat(df_chunks, ignore_index=True)
print('Processing finished.')

In [ ]:
myLabs

In [ ]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(myLabs, 'labresultoffset', 'labname', 'labresult')

In [ ]:
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'lab', 'labname', 'num_values')

In [ ]:
[x for x in myPredictorsDf.columns if 'first' in x]

In [ ]:
myPredictorsDf = myPredictorsDf1

In [ ]:
myPredictorsDf = myPredictorsDf.merge(patients_df[['patientunitstayid', 'hospitaldischargestatus']], on=['patientunitstayid'])

In [ ]:
myPredictorsDf['DeathAtDischarge'] = myPredictorsDf.hospitaldischargestatus == 'Expired'
myPredictorsDf['DeathAtDischarge'] = myPredictorsDf['DeathAtDischarge'].astype(int)

In [ ]:
myPredictorsDf['LastGCS15'] = (myPredictorsDf['LastGCS'] == '15').astype(int)

In [ ]:
myPredictorsDf.loc[myPredictorsDf.age == '> 89', 'age'] = 89

In [ ]:
myPredictorsDf.age = myPredictorsDf.fillna(0).age.astype(int)

In [ ]:
myPredictorsDf = myPredictorsDf[myPredictorsDf.age >= 18]

In [ ]:
myPredictorsDf

In [ ]:
# myPredictorsDf.to_csv('eICUPredictorsDiag.csv', index=False)

In [ ]:
myPredictorsDf.FirstMGCS

In [ ]:
['SUBJID',
 'groupe',
 'CPC_SC3',
 'J0_SEX',
 'J0_TAILLE',
 'J0_POIDS',
 'J0_BMI',
 'J0_AGE',
 'J0_PAS',
 'J0_PAD',
 'J0_PAM',
 'J0_FC',
 'J0_SPO2',
 'J0_GLASGOW',
 'J0_GLASGOW_CONTROLE',
 'J0_OCULAIRE',
 'J0_VERBALE',
 'J0_MOTRICE',
 'J0_PUPILG',
 'J0_PUPILG_REA',
 'J0_PUPILD',
 'J0_PUPILD_REA',
 'J0_CILIAIRE',
 'J0_CORNEEN',
 'J0_REFLEXVEST',
 'J0_REFLEXCEPH',
 'J0_REFLEXCARD',
 'J0_TEMP',
 'J0_IGSII',
 'J0_MCCABE',
 'J0_KNAUS',
 'J0_CHARLSON1',
 'J0_CHARLSON2',
 'J0_CHARLSON3',
 'J0_CHARLSON4',
 'J0_CHARLSON5',
 'J0_CHARLSON6',
 'J0_CHARLSON7',
 'J0_CHARLSON8',
 'J0_CHARLSON9',
 'J0_CHARLSON10',
 'J0_CHARLSON11',
 'J0_CHARLSON12',
 'J0_CHARLSON13',
 'J0_CHARLSON14',
 'V0_CHARLSON15',
 'V0_CHARLSON16',
 'V0_CHARLSON17',
 'V0_CHARLSON18',
 'V0_CHARLSON18B',
 'V0_CHARLSON19',
 'J0_CHARLSON',
 'J0_ATCD',
 'J0_CARDIO',
 'J0_NYHA',
 'J0_MYOCARD',
 'J0_ARTERIO',
 'J0_HTA',
 'J0_POUMON',
 'J0_IRC',
 'J0_HYPERCAP',
 'J0_O2',
 'J0_TABAC',
 'J0_RYTHM',
 'J0_CAUSE2_ACR',
 'J0_DSA',
 'J0_DSA_P',
 'J0_TEMOIN',
 'J0_TEMOIN_MASSE',
 'J0_LIEU_ACR',
 'J0_NOFLOW',
 'J0_LOWFLOW',
 'J0_ADRE',
 'J0_ADRE_DOS',
 'J0_CORDA',
 'J0_CORDA_DOS',
 'J0_BICARB',
 'J0_BICARB_DOS',
 'V0_REFROIDI',
 'V0_CHOC_AV',
 'V0_CHOC_AP',
 'V0_PLANCHE',
 'V0_THROMBO',
 'V0_CORO_ACR',
 'V0_ANGIO_ACR',
 'V0_ANGIO_YES',
 'V0_BALLON',
 'V0_ACR2',
 'J0_CURAR',
 'J0_SEDATIF',
 'J0_MORPHIN',
 'J0_COAGUL',
 'J0_AGREG',
 'J0_ANTIBIO',
 'J0_AMINE',
 'J0_NORA',
 'J0_ADRE2',
 'J0_DOBU',
 'J0_DOPA',
 'J0_PEP',
 'J0_FIO2',
 'J0_VT',
 'J0_FR',
 'BIO_LEUCO',
 'BIO_HEMO',
 'BIO_PLAQ',
 'BIO_TP',
 'BIO_DDIMERE',
 'BIO_SODIUM',
 'BIO_POTAS',
 'BIO_UREE',
 'BIO_CREAT',
 'BIO_CALCIUM',
 'BIO_MAGNE',
 'BIO_GLYCEMI',
 'BIO_PROTID',
 'BIO_LIPAS',
 'BIO_TROPO',
 'BIO_TEMP',
 'BIO_FIO2',
 'BIO_PH',
 'BIO_PAO2',
 'BIO_PACO2',
 'BIO_BICARB',
 'BIO_LACTAT',
 'BIO_TROPO2',
 'BIO_TROPO_CGT',
 'ECG',
 'ECG_QTC',
 'ECG_ANOMALI',
 'ECG_SUS_ST',
 'ECG_SOUS_ST',
 'ECG_BAVI',
 'ECG_BAVII',
 'ECG_BAVIII',
 'ECG_BBG',
 'ECG_BBD',
 'ECG_TACHICARD',
 'ECG_FIBRIL',
 'ECG_SALV_VENT',
 'ECG_FLUTER',
 'ECG_SALV_SUPRA',
 'SOFA_SC',
 'SOFA_RESPIR',
 'SOFA_CARDIO',
 'SOFA_COAG',
 'SOFA_NEURO',
 'SOFA_FOIE',
 'SOFA_RENAL',
 'EI_EI',
 'EI_HEMOSEVER',
 'EI_TRANSFUS',
 'EI_INTRACER',
 'EI_CHIR',
 'EI_EXTRARENAL',
 'EI_OAP',
 'EI_ECHO',
 'EI_DIURETIQ',
 'EI_CONVULS',
 'EI_ARYTHMI',
 'EI_ANTIEPILEPTIQ',
 'BARTHEL_SC',
 'SOFA_SC7',
 'SOFA_SC1',
 'DS_DC',
 'DAYS_ALIVE_30',
 'CPC12',
 'SEX']

In [ ]:
columns = [
    'patientunitstayid',
    'gender',
    'age',
    'hospitaladmittime24',
    'bmi',
    'nurse_first_Non-Invasive BP Diastolic',
    'nurse_first_Non-Invasive BP Systolic',
    'nurse_first_Non-Invasive BP Mean',
    'nurse_first_Heart Rate',
    'nurse_first_O2 Saturation',
    'nurse_first_Temperature (C)',
    'nurse_first_GCS Total',
    'nurse_last_GCS Total',
    'nurse_max_GCS Total',
    'nurse_min_GCS Total',
    'nurse_mean_GCS Total',
    'FirstGCSTime',
    'FirstGCS',
    'LastGCSTime',
    'LastGCS',
    'FirstMGCSTime',
    'FirstMGCS',
    'LastMGCSTime',
    'LastMGCS',
    'LastGCS15',
    'nurse_first_Motor',
    'diagnosis_initial rhythm: ventricular fibrillation',
    'diagnosis_ventricular fibrillation',
    'diagnosis_ventricular tachycardia',
    'diagnosis_initial rhythm: ventricular tachycardia',
    'diagnosis_initial rhythm: pulseless electrical activity',
    'diagnosis_initial rhythm: asystole',
]

In [ ]:
[x for x in myPredictorsDf.columns if 'offset' in x.lower()]

In [ ]:
myPredictorsDf[myPredictorsDf.treatment_hypothermia_x == 1].treatmentoffset

In [ ]:
myPredictorsDf['bmi'] = myPredictorsDf.admissionweight / (myPredictorsDf.admissionheight/100)**2

In [ ]:
myPredictorsDf.bmi